In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

Saving qa_df_resultados_flan_xl.csv to qa_df_resultados_flan_xl.csv
Saving qa_df_resultados_gemma2.csv to qa_df_resultados_gemma2.csv
Saving qa_df_resultados_large.csv to qa_df_resultados_large.csv
Saving qa_df_resultados_qwen15.csv to qa_df_resultados_qwen15.csv
Saving qa_df_resultados_tiny.csv to qa_df_resultados_tiny.csv


CLASSICS METRICS: ROUGE I BLEU

In [ ]:
!pip install rouge_score sacrebleu evaluate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=db4e68cf69218aa15c4a016e130d14b9a8b6a9a91def0e09c482dcc5ac7c39de
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
import evaluate
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import pandas as pd
from rouge_score import rouge_scorer
import evaluate
from google.colab import files

# BLEU metric sentence level
bleu = evaluate.load("bleu")

files_list = [
    "qa_df_resultados_flan_xl.csv",
    "qa_df_resultados_gemma2.csv",
    "qa_df_resultados_large.csv",
    "qa_df_resultados_qwen15.csv",
    "qa_df_resultados_tiny.csv"
]

combinaciones = [
    ("respuesta_con_rag", "SA"),
    ("respuesta_con_rag", "LA"),
    ("respuesta_sin_rag", "SA"),
    ("respuesta_sin_rag", "LA")
]

# ROUGE scorer
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

for file in files_list:

    print(f"\n==================== {file} ====================")

    qa_df = pd.read_csv(file)

    # limpiar nombres de columnas
    qa_df.columns = qa_df.columns.str.strip()

    # asegurar strings
    for col in ["respuesta_con_rag", "respuesta_sin_rag", "SA", "LA"]:
        qa_df[col] = qa_df[col].astype(str)

    # dataframe limpio SOLO con primeras 4 columnas
    result_df = qa_df.iloc[:, :4].copy()

    for gen_col, ref_col in combinaciones:

        rouge1_f1, rouge1_p, rouge1_r = [], [], []
        rouge2_f1, rouge2_p, rouge2_r = [], [], []
        rougeL_f1, rougeL_p, rougeL_r = [], [], []
        bleu_scores = []

        # cálculo por fila
        for gen, ref in zip(qa_df[gen_col], qa_df[ref_col]):

            # ROUGE
            r = scorer.score(ref, gen)

            rouge1_f1.append(r["rouge1"].fmeasure)
            rouge1_p.append(r["rouge1"].precision)
            rouge1_r.append(r["rouge1"].recall)

            rouge2_f1.append(r["rouge2"].fmeasure)
            rouge2_p.append(r["rouge2"].precision)
            rouge2_r.append(r["rouge2"].recall)

            rougeL_f1.append(r["rougeL"].fmeasure)
            rougeL_p.append(r["rougeL"].precision)
            rougeL_r.append(r["rougeL"].recall)

            # BLEU
            b = bleu.compute(
                predictions=[gen],
                references=[[ref]]
            )

            bleu_scores.append(b["bleu"])

        # guardar métricas en dataframe limpio
        result_df[f"rouge1_f1_{gen_col}_{ref_col}"] = rouge1_f1
        result_df[f"rouge1_precision_{gen_col}_{ref_col}"] = rouge1_p
        result_df[f"rouge1_recall_{gen_col}_{ref_col}"] = rouge1_r

        result_df[f"rouge2_f1_{gen_col}_{ref_col}"] = rouge2_f1
        result_df[f"rouge2_precision_{gen_col}_{ref_col}"] = rouge2_p
        result_df[f"rouge2_recall_{gen_col}_{ref_col}"] = rouge2_r

        result_df[f"rougeL_f1_{gen_col}_{ref_col}"] = rougeL_f1
        result_df[f"rougeL_precision_{gen_col}_{ref_col}"] = rougeL_p
        result_df[f"rougeL_recall_{gen_col}_{ref_col}"] = rougeL_r

        result_df[f"bleu_{gen_col}_{ref_col}"] = bleu_scores

        # mostrar medias
        print(f"\n=== {gen_col} vs {ref_col} ===")
        print(f"ROUGE-1 F1: {sum(rouge1_f1)/len(rouge1_f1):.3f}")
        print(f"ROUGE-2 F1: {sum(rouge2_f1)/len(rouge2_f1):.3f}")
        print(f"ROUGE-L F1: {sum(rougeL_f1)/len(rougeL_f1):.3f}")
        print(f"BLEU:       {sum(bleu_scores)/len(bleu_scores):.3f}")

    # nombre del modelo
    model_name = file.replace("qa_df_resultados_", "").replace(".csv", "")

    # nombre output
    output_file = f"{model_name}_classic_metrics.csv"

    # guardar csv
    result_df.to_csv(output_file, index=False)

    print(f"\n Guardado: {output_file}")

    files.download(output_file)


==================== qa_df_resultados_flan_xl.csv ====================

=== respuesta_con_rag vs SA ===
ROUGE-1 F1: 0.630
ROUGE-2 F1: 0.343
ROUGE-L F1: 0.619
BLEU:       0.148

=== respuesta_con_rag vs LA ===
ROUGE-1 F1: 0.342
ROUGE-2 F1: 0.222
ROUGE-L F1: 0.328
BLEU:       0.101

=== respuesta_sin_rag vs SA ===
ROUGE-1 F1: 0.145
ROUGE-2 F1: 0.016
ROUGE-L F1: 0.140
BLEU:       0.000

=== respuesta_sin_rag vs LA ===
ROUGE-1 F1: 0.087
ROUGE-2 F1: 0.016
ROUGE-L F1: 0.079
BLEU:       0.000

✔ Guardado: flan_xl_classic_metrics.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


==================== qa_df_resultados_gemma2.csv ====================

=== respuesta_con_rag vs SA ===
ROUGE-1 F1: 0.346
ROUGE-2 F1: 0.220
ROUGE-L F1: 0.327
BLEU:       0.121

=== respuesta_con_rag vs LA ===
ROUGE-1 F1: 0.672
ROUGE-2 F1: 0.462
ROUGE-L F1: 0.592
BLEU:       0.296

=== respuesta_sin_rag vs SA ===
ROUGE-1 F1: 0.102
ROUGE-2 F1: 0.016
ROUGE-L F1: 0.086
BLEU:       0.001

=== respuesta_sin_rag vs LA ===
ROUGE-1 F1: 0.302
ROUGE-2 F1: 0.123
ROUGE-L F1: 0.237
BLEU:       0.040

✔ Guardado: gemma2_classic_metrics.csv


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


==================== qa_df_resultados_tiny.csv ====================

=== respuesta_con_rag vs SA ===
ROUGE-1 F1: 0.183
ROUGE-2 F1: 0.107
ROUGE-L F1: 0.166
BLEU:       0.046

=== respuesta_con_rag vs LA ===
ROUGE-1 F1: 0.397
ROUGE-2 F1: 0.255
ROUGE-L F1: 0.342
BLEU:       0.130

=== respuesta_sin_rag vs SA ===
ROUGE-1 F1: 0.097
ROUGE-2 F1: 0.023
ROUGE-L F1: 0.082
BLEU:       0.002

=== respuesta_sin_rag vs LA ===
ROUGE-1 F1: 0.284
ROUGE-2 F1: 0.131
ROUGE-L F1: 0.229
BLEU:       0.038

✔ Guardado: tiny_classic_metrics.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#courpus level
import pandas as pd
import evaluate
from google.colab import files

# Cargar BLEU
bleu = evaluate.load("bleu")

files_list = [
    "qa_df_resultados_flan_xl.csv",
    "qa_df_resultados_gemma2.csv",
    "qa_df_resultados_large.csv",
    "qa_df_resultados_qwen15.csv",
    "qa_df_resultados_tiny.csv"
]

combinaciones = [
    ("respuesta_con_rag", "SA"),
    ("respuesta_con_rag", "LA"),
    ("respuesta_sin_rag", "SA"),
    ("respuesta_sin_rag", "LA")
]

for file in files_list:
    print(f"\n==================== {file} ====================")

    qa_df = pd.read_csv(file)
    qa_df.columns = qa_df.columns.str.strip()

    for col in ["respuesta_con_rag", "respuesta_sin_rag", "SA", "LA"]:
        qa_df[col] = qa_df[col].astype(str)

    for gen_col, ref_col in combinaciones:

        # Listas completas de predicciones y referencias
        predictions = qa_df[gen_col].tolist()
        references = [[r] for r in qa_df[ref_col].tolist()]  # formato para BLEU

        # BLEU a nivel de corpus (acumula todos los n-gramas)
        b = bleu.compute(predictions=predictions, references=references)

        print(f"\n=== {gen_col} vs {ref_col} ===")
        print(f"BLEU (corpus-level): {b['bleu']:.4f}")


==================== qa_df_resultados_flan_xl.csv ====================

=== respuesta_con_rag vs SA ===
BLEU (corpus-level): 0.3680

=== respuesta_con_rag vs LA ===
BLEU (corpus-level): 0.1159

=== respuesta_sin_rag vs SA ===
BLEU (corpus-level): 0.0000

=== respuesta_sin_rag vs LA ===
BLEU (corpus-level): 0.0000

==================== qa_df_resultados_gemma2.csv ====================

=== respuesta_con_rag vs SA ===
BLEU (corpus-level): 0.1581

=== respuesta_con_rag vs LA ===
BLEU (corpus-level): 0.3374

=== respuesta_sin_rag vs SA ===
BLEU (corpus-level): 0.0040

=== respuesta_sin_rag vs LA ===
BLEU (corpus-level): 0.0629

==================== qa_df_resultados_large.csv ====================

=== respuesta_con_rag vs SA ===
BLEU (corpus-level): 0.3084

=== respuesta_con_rag vs LA ===
BLEU (corpus-level): 0.1705

=== respuesta_sin_rag vs SA ===
BLEU (corpus-level): 0.0000

=== respuesta_sin_rag vs LA ===
BLEU (corpus-level): 0.0000

==================== qa_df_resultados_qwen15.csv =====

RAGAS

In [ ]:
# Importar
from google.colab import files
import pandas as pd

uploaded = files.upload()

Saving qa_df_resultados_flan_xl.csv to qa_df_resultados_flan_xl (1).csv
Saving qa_df_resultados_gemma2.csv to qa_df_resultados_gemma2 (1).csv
Saving qa_df_resultados_large.csv to qa_df_resultados_large (1).csv
Saving qa_df_resultados_qwen15.csv to qa_df_resultados_qwen15 (1).csv
Saving qa_df_resultados_tiny.csv to qa_df_resultados_tiny (1).csv


In [ ]:
!pip install langchain-core==0.3.0 langchain-openai==0.2.0 langchain-community==0.3.0 ragas==0.1.14 nest-asyncio==1.6.0

In [ ]:
!pip install --upgrade ragas

In [ ]:
#cargar los ficheros uno a uno
qa_df = pd.read_csv("qa_df_resultados_large.csv")
# df RAG
df_ragas = qa_df[['Q', 'respuesta_con_rag', 'SA', 'LA',
                  'chunk_rec_1_texto', 'chunk_rec_2_texto', 'chunk_rec_3_texto']].copy()

df_ragas = df_ragas.rename(columns={
    'Q': 'question',
    'respuesta_con_rag': 'answer'
})

def crear_lista_contextos(fila):
    chunks = []
    for col in ['chunk_rec_1_texto', 'chunk_rec_2_texto', 'chunk_rec_3_texto']:
        texto = fila[col]
        if pd.notna(texto) and texto != "":
            chunks.append(texto)
    return chunks if chunks else [""]   # Lista con string vacío si no hay chunks

df_ragas['contexts'] = df_ragas.apply(crear_lista_contextos, axis=1)

# Seleccionar columnas finales
df_ragas = df_ragas[['question', 'answer', 'contexts', 'SA', 'LA']]

# df no RAG
df_ragas_norag = qa_df[['Q', 'respuesta_sin_rag', 'SA', 'LA']].copy()
df_ragas_norag = df_ragas_norag.rename(columns={
    'Q': 'question',
    'respuesta_sin_rag': 'answer'
})
df_ragas_norag = df_ragas_norag[['question', 'answer', 'SA', 'LA']]

In [ ]:
# @title
# RAG
import os
import asyncio
import nest_asyncio
import pandas as pd
from tqdm.asyncio import tqdm
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.metrics.collections import Faithfulness, AnswerRelevancy, AnswerCorrectness
from google.colab import userdata, files

nest_asyncio.apply()

# Configuración
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
client = AsyncOpenAI()
llm = llm_factory("gpt-4o-mini", client=client, temperature=0)
embeddings = embedding_factory("openai", model="text-embedding-3-small", client=client)

faithfulness_scorer = Faithfulness(llm=llm)
relevancy_scorer = AnswerRelevancy(llm=llm, embeddings=embeddings)
correctness_scorer = AnswerCorrectness(llm=llm, embeddings=embeddings)

async def evaluar_fila_rag(row):
    question = row['question']
    answer = row['answer']
    contexts = row['contexts']

    # Faithfulness
    f = await faithfulness_scorer.ascore(
        user_input=question,
        response=answer,
        retrieved_contexts=contexts
    )
    faithfulness_score = f.value

    # Answer Relevancy
    ar = await relevancy_scorer.ascore(
        user_input=question,
        response=answer
    )
    relevancy_score = ar.value

    # Answer Correctness para SA y LA
    ac_sa = await correctness_scorer.ascore(
        user_input=question,
        response=answer,
        reference=row['SA']
    )
    ac_la = await correctness_scorer.ascore(
        user_input=question,
        response=answer,
        reference=row['LA']
    )

    return {
        'question': question,
        'answer': answer,
        'faithfulness': faithfulness_score,
        'answer_relevancy': relevancy_score,
        'answer_correctness_SA': ac_sa.value,
        'answer_correctness_LA': ac_la.value,
        'SA': row['SA'],
        'LA': row['LA']
    }

async def evaluar_dataset_rag(df, max_concurrent=5):
    semaphore = asyncio.Semaphore(max_concurrent)
    async def limitada(row):
        async with semaphore:
            return await evaluar_fila_rag(row)
    tareas = [limitada(row) for _, row in df.iterrows()]
    resultados = await tqdm.gather(*tareas, desc="Evaluando RAG")
    return pd.DataFrame(resultados)

ModuleNotFoundError: No module named 'ragas'

In [ ]:
print("Evaluando RAG (SA y LA a la vez)...")
df_rag_result = await evaluar_dataset_rag(df_ragas, max_concurrent=5)
df_rag_result.to_csv("resultados_RAG_large.csv", index=False)
files.download("resultados_RAG_large.csv")
print("✅ Evaluación RAG finalizada.")

Evaluando RAG (SA y LA a la vez)...


Evaluando RAG: 100%|██████████| 136/136 [09:54<00:00,  4.37s/it]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Evaluación RAG finalizada.


In [ ]:
df_rag_result[['faithfulness', 'answer_relevancy', 'answer_correctness_SA', 'answer_correctness_LA']].describe()

,faithfulness,answer_relevancy,answer_correctness_SA,answer_correctness_LA
count,136.000000,136.000000,136.000000,136.000000
mean,0.834559,0.286586,0.704805,0.677674
std,0.344705,0.231408,0.299436,0.259083
min,0.000000,0.000000,0.052114,0.042191
25%,1.000000,0.122535,0.559706,0.544808
50%,1.000000,0.257251,0.836365,0.777368
75%,1.000000,0.413118,0.952458,0.883068
max,1.000000,0.881420,1.000000,0.982137


In [ ]:
# @title
#NO RAG
import os
import asyncio
import nest_asyncio
import pandas as pd
from tqdm.asyncio import tqdm
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.metrics.collections import AnswerRelevancy, AnswerCorrectness
from google.colab import userdata, files

nest_asyncio.apply()

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
client = AsyncOpenAI()

llm = llm_factory("gpt-4o-mini", client=client, temperature=0)
embeddings = embedding_factory("openai", model="text-embedding-3-small", client=client)

relevancy_scorer = AnswerRelevancy(llm=llm, embeddings=embeddings)
correctness_scorer = AnswerCorrectness(llm=llm, embeddings=embeddings)

async def evaluar_fila_norag(row):
    question = row['question']
    answer = row['answer']

    # Answer Relevancy
    ar = await relevancy_scorer.ascore(
        user_input=question,
        response=answer
    )
    relevancy_score = ar.value

    # Answer Correctness para SA y LA
    ac_sa = await correctness_scorer.ascore(
        user_input=question,
        response=answer,
        reference=row['SA']
    )
    ac_la = await correctness_scorer.ascore(
        user_input=question,
        response=answer,
        reference=row['LA']
    )

    return {
        'question': question,
        'answer': answer,
        'answer_relevancy': relevancy_score,
        'answer_correctness_SA': ac_sa.value,
        'answer_correctness_LA': ac_la.value,
        'SA': row['SA'],
        'LA': row['LA']
    }

async def evaluar_dataset_norag(df, max_concurrent=5):
    semaphore = asyncio.Semaphore(max_concurrent)
    async def limitada(row):
        async with semaphore:
            return await evaluar_fila_norag(row)
    tareas = [limitada(row) for _, row in df.iterrows()]
    resultados = await tqdm.gather(*tareas, desc="Evaluando NoRAG")
    return pd.DataFrame(resultados)

In [ ]:
print("Evaluando NoRAG (SA y LA a la vez)...")
df_norag_result = await evaluar_dataset_norag(df_ragas_norag, max_concurrent=5)
df_norag_result.to_csv("resultados_NoRAG_large.csv", index=False)
files.download("resultados_NoRAG_large.csv")
print("✅ Evaluación NoRAG finalizada.")

Evaluando NoRAG (SA y LA a la vez)...


Evaluando NoRAG: 100%|██████████| 136/136 [10:25<00:00,  4.60s/it]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Evaluación NoRAG finalizada.


In [ ]:
df_norag_result[['answer_relevancy', 'answer_correctness_SA', 'answer_correctness_LA']].describe()

,answer_relevancy,answer_correctness_SA,answer_correctness_LA
count,136.000000,136.000000,136.000000
mean,0.632503,0.166072,0.273422
std,0.412235,0.167543,0.186802
min,0.000000,0.023973,0.025242
25%,0.000000,0.070057,0.154099
50%,0.854491,0.104582,0.185405
75%,0.949759,0.154907,0.414817
max,1.000000,0.965850,0.965826


In [ ]:
# @title
!pip show ragas | grep "Location:"

In [ ]:
# @title
!cat /usr/local/lib/python3.12/dist-packages/ragas/metrics/_answer_correctness.py